In [ ]:
//#r "../../src/L4-application/BoSSSpad/bin/Release/net5.0/BoSSSpad.dll"
//#r "../../src/L4-application/BoSSSpad/bin/Debug/net5.0/BoSSSpad.dll"
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
using MPI.Wrappers;
using NUnit.Framework;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Solution.XdgTimestepping;

In [ ]:
BoSSSshell.WorkflowMgm.Init("RotatingDiskBoundaryLayer2D");

In [ ]:
ExecutionQueues

index,type,DeploymentBaseDirectory,DeployRuntime,RuntimeLocation,Name,DotnetRuntime,BatchInstructionDir,AllowedDatabasesPaths,Username,ServerName,ComputeNodes,DefaultJobPriority,SingleNode
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,D:\local\binaries,False,<null>,LocalPC,dotnet,<null>,[ D:\local\ ],,,,,
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,\\hpccluster\hpccluster-scratch\smuda\binaries,True,win\amd64,FDY-WindowsHPC,dotnet,,[ \\hpccluster\hpccluster-scratch\smuda\ ],FDY\smuda,DC2,<null>,Normal,True


In [ ]:
var myBatch = ExecutionQueues[1];
myBatch

RuntimeLocation,DeploymentBaseDirectory,DeployRuntime,Name,DotnetRuntime,Username,ServerName,ComputeNodes,DefaultJobPriority,SingleNode,AllowedDatabasesPaths
win\amd64,\\hpccluster\hpccluster-scratch\smuda\binaries,True,FDY-WindowsHPC,dotnet,FDY\smuda,DC2,<null>,Normal,True,[ \\hpccluster\hpccluster-scratch\smuda\ ]


## grid creation

In [ ]:
static public Grid3D RotatingDisk2DSlice_NormalToAzimuthal(double radiusOP, double l_radial, double l_azimuthal, double h_axial, int res_azimuthal, int res_radial, int res_axial) {

    double[] xNodes = GenericBlas.Linspace(-l_radial / 2.0, l_radial / 2.0, res_radial + 1);    // radial direction
    double[] yNodes = GenericBlas.Linspace(-l_azimuthal / 2.0, l_azimuthal / 2.0, res_azimuthal + 1);    // azimuthal direction
    double[] zNodes = GenericBlas.Linspace(0.0, h_axial, res_axial + 1);    // axial direction
    
    var grd = Grid3D.Cartesian3DGrid(xNodes, yNodes, zNodes, periodicY: false);
    grd.Name = $"RotatingDiskSector3D_Linearized_{res_radial}x{res_azimuthal}x{res_axial}";

    grd.EdgeTagNames.Add(1, "wall_rotatingDisk");
    // grd.EdgeTagNames.Add(2, "velocity_inlet_top");
    grd.EdgeTagNames.Add(2, "freeslip_top");
    // grd.EdgeTagNames.Add(3, "velocity_inlet_front");
    grd.EdgeTagNames.Add(3, "freeslip_front");
    // grd.EdgeTagNames.Add(4, "velocity_inlet_back");
    grd.EdgeTagNames.Add(4, "freeslip_back");
    grd.EdgeTagNames.Add(5, "velocity_inlet_upstream");
    // grd.EdgeTagNames.Add(6, "velocity_inlet_downstream");
    grd.EdgeTagNames.Add(6, "pressure_outlet_downstream");

    grd.DefineEdgeTags(delegate (Vector X) {
        byte et = 0;
        if (X.z.Abs() <= 1e-8)
            et = 1;
        if ((X.z - h_axial).Abs() <= 1e-8)
            et = 2;
        if ((X.x + (l_radial / 2.0)).Abs() <= 1e-8)
            et = 3;
        if ((X.x - (l_radial / 2.0)).Abs() <= 1e-8)
            et = 4;
        if ((X.y + (l_azimuthal / 2.0)).Abs() <= 1e-8)
            et = 5;
        if ((X.y - (l_azimuthal / 2.0)).Abs() <= 1e-8)
            et = 6;

        return et;
    });    

    return grd;
}

In [ ]:
static public Grid2D RotatingDisk2DSlice_NormalToRadial(double l_azimuthal, double h_axial, int res_azimuthal, int res_axial) {

    double[] xNodes = GenericBlas.Linspace(-l_azimuthal / 2.0, l_azimuthal / 2.0, res_azimuthal + 1);    // azimuthal direction
    double[] yNodes = GenericBlas.Linspace(0.0, h_axial, res_axial + 1);    // axial direction
    // double[] yNodes = GenericBlas.Logspace(0.0, h_axial, res_axial + 1);    // axial direction
    
    var grd = Grid2D.Cartesian2DGrid(xNodes, yNodes, periodicY: false);
    grd.Name = $"RotatingDiskSector3D_Linearized_{res_azimuthal}x{res_axial}";

    grd.EdgeTagNames.Add(1, "wall_rotatingDisk");
    grd.EdgeTagNames.Add(2, "pressure_dirichlet_top");
    grd.EdgeTagNames.Add(3, "velocity_inlet_upstream");
    // grd.EdgeTagNames.Add(4, "pressure_outlet_downstream");
    grd.EdgeTagNames.Add(4, "pressure_dirichlet_downstream");
    // grd.EdgeTagNames.Add(4, "velocity_inlet_downstream");


    grd.DefineEdgeTags(delegate (Vector X) {
        byte et = 0;
        if (X.y.Abs() <= 1e-8)
            et = 1;
        if ((X.y - h_axial).Abs() <= 1e-8)
            et = 2;
        if ((X.x + (l_azimuthal / 2.0)).Abs() <= 1e-8)
            et = 3;
        if ((X.x - (l_azimuthal / 2.0)).Abs() <= 1e-8)
            et = 4;

        return et;
    });    

    return grd;
}

## simulation setup

In [ ]:
double radiusOP = 100; // operating point -> Re = radiusOp / Lstar
double viscosity = 0.01; // kinematic viscosity
double omega = 0.01; // rotation rate
double Lstar = Math.Sqrt(viscosity / omega);
Lstar

1

In [ ]:
double z = 10;
double zstar = z * Lstar;
zstar

10

In [ ]:
double density = 1.0;

In [ ]:
Formula MovingWall = new Formula(
    "VelX",
    false,
    "double VelX(double[] X) { double radiusOP = 100.0; double omega = 0.01; return (0.0 + radiusOP) * omega; } "
);

### prescribed boundary conditions (initial conditions)

In [ ]:
using BoSSS.Application.XNSE_Solver.SpecificSolutions;

In [ ]:
MultidimensionalArray velU_P = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolutionPython_VelocityU.txt");
MultidimensionalArray velV_P = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolutionPython_VelocityV.txt");
MultidimensionalArray velW_P = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolutionPython_VelocityW.txt");
MultidimensionalArray pressureP_P = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolutionPython_PressureP.txt");

In [ ]:
MultidimensionalArray velU_M = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolutionMatlab_VelocityU.txt");
MultidimensionalArray velV_M = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolutionMatlab_VelocityV.txt");
MultidimensionalArray velW_M = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolutionMatlab_VelocityW.txt");
MultidimensionalArray pressureP_M = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolutionMatlab_PressureP.txt");

In [ ]:
velU_P.Lengths

index,value
0,4002
1,2


In [ ]:
Plot2Ddata plt = new Plot2Ddata();
var fmtP = new PlotFormat();
fmtP.LineColor = LineColors.Blue;
var fmtM = new PlotFormat();

var solP = velW_P;
var solM = velW_M;

plt.AddDataGroup(solP.ExtractSubArrayShallow(new int[] { -1, 1}).To1DArray(), solP.ExtractSubArrayShallow(new int[] { -1, 0}).To1DArray(), fmtP);
plt.AddDataGroup(solM.ExtractSubArrayShallow(new int[] { -1, 1}).To1DArray(), solM.ExtractSubArrayShallow(new int[] { -1, 0}).To1DArray(), fmtM);
var gp = plt.ToGnuplot();
gp.PlotNow()

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 5 
 
 
 
 
 10 
 
 
 
 
 15 
 
 
 
 
 20 
 
 
 
 
 -0.9 
 
 
 
 
 -0.8 
 
 
 
 
 -0.7 
 
 
 
 
 -0.6 
 
 
 
 
 -0.5 
 
 
 
 
 -0.4 
 
 
 
 
 -0.3 
 
 
 
 
 -0.2 
 
 
 
 
 -0.1 
 
 
 
 
 0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 gnuplot_plot_1 
 
	<path stroke='rgb( 0, 0, 255)' d='M750.1,564.0 L750.1,563.8 L750.1,563.7 L750.0,563.6 L749.9,563.4 L749.8,563.3 L749.7,563.2 L749.6,563.0
 L749.4,562.9 L749.3,562.8 L749.1,562.7 L748.9,562.5 L748.7,562.4 L748.4,562.3 L748.2,562.1 L747.9,562.0
 L747.6,561.9 L747.3,561.7 L747.0,561.6 L746.6,561.5 L746.3,561.3 L745.9,561.2 L745.5,561.1 L745.1,560.9
 L744.7,560.8 L744.3,560.7 L743.8,560.5 L743.3,560.4 L742.9,560.3 L742.4,560.1 L741.8,560.0 L741.3,559.9
 L740.8,559.7 L740.2,559.6 L739.6,559.5 L739.1,559.4 L738.5,559.2 L737.9,559.1 L737.2,559.0 L736.6,558.8
 L735.9,558.7 L735.3,558.6 L734.6,558.4 L733.9,558.3 L733.2,558.2 L732.5,558.0 L731.8,557.9 L731.0,557.8
 L730.3,557.6 L729.5,557.5 L728.7,557.4 L728.0,557.2 L727.2,557.1 L726.3,557.0 L725.5,556.8 L724.7,556.7
 L723.9,556.6 L723.0,556.4 L722.1,556.3 L721.3,556.2 L720.4,556.1 L719.5,555.9 L718.6,555.8 L717.7,555.7
 L716.7,555.5 L715.8,555.4 L714.9,555.3 L713.9,555.1 L712.9,555.0 L712.0,554.9 L711.0,554.7 L710.0,554.6
 L709.0,554.5 L708.0,554.3 L707.0,554.2 L706.0,554.1 L704.9,553.9 L703.9,553.8 L702.8,553.7 L701.8,553.5
 L700.7,553.4 L699.6,553.3 L698.6,553.1 L697.5,553.0 L696.4,552.9 L695.3,552.8 L694.2,552.6 L693.0,552.5
 L691.9,552.4 L690.8,552.2 L689.7,552.1 L688.5,552.0 L687.4,551.8 L686.2,551.7 L685.0,551.6 L683.9,551.4
 L682.7,551.3 L681.5,551.2 L680.3,551.0 L679.1,550.9 L677.9,550.8 L676.7,550.6 L675.5,550.5 L674.3,550.4
 L673.1,550.2 L671.8,550.1 L670.6,550.0 L669.4,549.8 L668.1,549.7 L666.9,549.6 L665.6,549.5 L664.3,549.3
 L663.1,549.2 L661.8,549.1 L660.5,548.9 L659.3,548.8 L658.0,548.7 L656.7,548.5 L655.4,548.4 L654.1,548.3
 L652.8,548.1 L651.5,548.0 L650.2,547.9 L648.9,547.7 L647.6,547.6 L646.3,547.5 L645.0,547.3 L643.6,547.2
 L642.3,547.1 L641.0,546.9 L639.6,546.8 L638.3,546.7 L637.0,546.6 L635.6,546.4 L634.3,546.3 L632.9,546.2
 L631.6,546.0 L630.2,545.9 L628.9,545.8 L627.5,545.6 L626.1,545.5 L624.8,545.4 L623.4,545.2 L622.0,545.1
 L620.7,545.0 L619.3,544.8 L617.9,544.7 L616.5,544.6 L615.1,544.4 L613.8,544.3 L612.4,544.2 L611.0,544.0
 L609.6,543.9 L608.2,543.8 L606.8,543.6 L605.4,543.5 L604.0,543.4 L602.6,543.3 L601.2,543.1 L599.8,543.0
 L598.4,542.9 L597.0,542.7 L595.6,542.6 L594.2,542.5 L592.8,542.3 L591.4,542.2 L590.0,542.1 L588.6,541.9
 L587.2,541.8 L585.8,541.7 L584.4,541.5 L583.0,541.4 L581.6,541.3 L580.2,541.1 L578.7,541.0 L577.3,540.9
 L575.9,540.7 L574.5,540.6 L573.1,540.5 L571.7,540.3 L570.3,540.2 L568.8,540.1 L567.4,540.0 L566.0,539.8
 L564.6,539.7 L563.2,539.6 L561.8,539.4 L560.4,539.3 L558.9,539.2 L557.5,539.0 L556.1,538.9 L554.7,538.8
 L553.3,538.6 L551.9,538.5 L550.5,538.4 L549.0,538.2 L547.6,538.1 L546.2,538.0 L544.8,537.8 L543.4,537.7
 L542.0,537.6 L540.6,537.4 L539.2,537.3 L537.8,537.2 L536.4,537.0 L534.9,536.9 L533.5,536.8 L532.1,536.7
 L530.7,536.5 L529.3,536.4 L527.9,536.3 L526.5,536.1 L525.1,536.0 L523.7,535.9 L522.3,535.7 L520.9,535.6
 L519.5,535.5 L518.1,535.3 L516.7,535.2 L515.3,535.1 L514.0,534.9 L512.6,534.8 L511.2,534.7 L509.8,534.5
 L508.4,534.4 L507.0,534.3 L505.6,534.1 L504.3,534.0 L502.9,533.9 L501.5,533.7 L500.1,533.6 L498.7,533.5
 L497.4,533.4 L496.0,533.2 L494.6,533.1 L493.3,533.0 L491.9,532.8 L490.5,532.7 L489.2,532.6 L487.8,532.4
 L486.4,532.3 L485.1,532.2 L483.7,532.0 L482.4,531.9 L481.0,531.8 L479.7,531.6 L478.3,531.5 L477.0,531.4
 L475.6,531.2 L474.3,531.1 L472.9,531.0 L471.6,530

In [ ]:
var velU = velU_P;
var velV = velV_P;
var velW = velW_P;
var pressureP = pressureP_P;

// var vonKarman_velU = new RotatingDiskBoundaryLayerConditions() { selectedSimilarityVariable = SimilarityVariable_vonKarmanFlow.velocityU };
// vonKarman_velU.SetData(velU.GetColumn(0), velU.GetColumn(1), radiusOP, omega, viscosity, density);
var vonKarman_velV = new RotatingDiskBoundaryLayerConditions() { selectedSimilarityVariable = SimilarityVariable_vonKarmanFlow.velocityV };
vonKarman_velV.SetData(velV.GetColumn(0), velV.GetColumn(1), radiusOP, omega, viscosity, density, false, -1, 0, 1);
var vonKarman_velW = new RotatingDiskBoundaryLayerConditions() { selectedSimilarityVariable = SimilarityVariable_vonKarmanFlow.velocityW };
vonKarman_velW.SetData(velW.GetColumn(0), velW.GetColumn(1), radiusOP, omega, viscosity, density, false, -1, 0, 1);
var vonKarman_P = new RotatingDiskBoundaryLayerConditions() { selectedSimilarityVariable = SimilarityVariable_vonKarmanFlow.pressureP };
vonKarman_P.SetData(pressureP.GetColumn(0), pressureP.GetColumn(1), radiusOP, omega, viscosity, density, false, -1, 0, 1);

In [ ]:
int numVal = 1000;
double stepSize = zstar / numVal; 
double[] zValues = Enumerable.Range(0, numVal+1).Select(idx => idx * stepSize).ToArray();
double[][] varValues = new double[4][];
varValues[0] = new double[numVal+1];
varValues[1] = new double[numVal+1];
varValues[2] = new double[numVal+1];

string[] varNames = new string[] {"VelocityX", "VelocityY", "Pressure"};
for (int i = 0; i <= numVal; i++) {
    // varValues[0][i] = vonKarman_velU.Evaluate(new double[] {0.0, zValues[i]}, 0.0);
    varValues[0][i] = vonKarman_velV.Evaluate(new double[] {0.0, zValues[i]}, 0.0);
    varValues[1][i] = vonKarman_velW.Evaluate(new double[] {0.0, zValues[i]}, 0.0);
    varValues[2][i] = vonKarman_P.Evaluate(new double[] {0.0, zValues[i]}, 0.0);
}

In [ ]:
Plot2Ddata plt = new Plot2Ddata();
var fmt = new PlotFormat();

int idx = 0;
plt.AddDataGroup(varNames[idx], varValues[idx], zValues, fmt);
var gp = plt.ToGnuplot();
gp.PlotNow()

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 2 
 
 
 
 
 4 
 
 
 
 
 6 
 
 
 
 
 8 
 
 
 
 
 10 
 
 
 
 
 0 
 
 
 
 
 0.1 
 
 
 
 
 0.2 
 
 
 
 
 0.3 
 
 
 
 
 0.4 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 0.7 
 
 
 
 
 0.8 
 
 
 
 
 0.9 
 
 
 
 
 1 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 VelocityX 
 
 
 VelocityX 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M577.4,57.1 L630.8,57.1 M750.1,564.0 L745.8,563.5 L741.4,562.9 L737.1,562.4 L732.8,561.9 L728.4,561.4
 L724.1,560.8 L719.8,560.3 L715.4,559.8 L711.1,559.2 L706.8,558.7 L702.5,558.2 L698.2,557.7 L693.9,557.1
 L689.7,556.6 L685.4,556.1 L681.1,555.6 L676.9,555.0 L672.6,554.5 L668.4,554.0 L664.2,553.4 L659.9,552.9
 L655.7,552.4 L651.6,551.9 L647.4,551.3 L643.2,550.8 L639.1,550.3 L634.9,549.7 L630.8,549.2 L626.7,548.7
 L622.6,548.2 L618.5,547.6 L614.5,547.1 L610.4,546.6 L606.4,546.1 L602.3,545.5 L598.3,545.0 L594.4,544.5
 L590.4,543.9 L586.4,543.4 L582.5,542.9 L578.6,542.4 L574.7,541.8 L570.8,541.3 L566.9,540.8 L563.1,540.2
 L559.2,539.7 L555.4,539.2 L551.6,538.7 L547.8,538.1 L544.1,537.6 L540.4,537.1 L536.6,536.5 L532.9,536.0
 L529.2,535.5 L525.6,535.0 L521.9,534.4 L518.3,533.9 L514.7,533.4 L511.1,532.9 L507.6,532.3 L504.0,531.8
 L500.5,531.3 L497.0,530.7 L493.5,530.2 L490.0,529.7 L486.6,529.2 L483.2,528.6 L479.7,528.1 L476.4,527.6
 L473.0,527.0 L469.6,526.5 L466.3,526.0 L463.0,525.5 L459.7,524.9 L456.5,524.4 L453.2,523.9 L450.0,523.4
 L446.8,522.8 L443.6,522.3 L440.4,521.8 L437.3,521.2 L434.2,520.7 L431.1,520.2 L428.0,519.7 L424.9,519.1
 L421.9,518.6 L418.9,518.1 L415.9,517.5 L412.9,517.0 L409.9,516.5 L407.0,516.0 L404.1,515.4 L401.2,514.9
 L398.3,514.4 L395.4,513.8 L392.6,513.3 L389.7,512.8 L386.9,512.3 L384.1,511.7 L381.4,511.2 L378.6,510.7
 L375.9,510.2 L373.2,509.6 L370.5,509.1 L367.8,508.6 L365.2,508.0 L362.6,507.5 L360.0,507.0 L357.4,506.5
 L354.8,505.9 L352.2,505.4 L349.7,504.9 L347.2,504.3 L344.7,503.8 L342.2,503.3 L339.7,502.8 L337.3,502.2
 L334.9,501.7 L332.5,501.2 L330.1,500.7 L327.7,500.1 L325.3,499.6 L323.0,499.1 L320.7,498.5 L318.4,498.0
 L316.1,497.5 L313.8,497.0 L311.6,496.4 L309.3,495.9 L307.1,495.4 L304.9,494.8 L302.7,494.3 L300.6,493.8
 L298.4,493.3 L296.3,492.7 L294.2,492.2 L292.1,491.7 L290.0,491.1 L287.9,490.6 L285.9,490.1 L283.9,489.6
 L281.8,489.0 L279.8,488.5 L277.9,488.0 L275.9,487.5 L273.9,486.9 L272.0,486.4 L270.1,485.9 L268.2,485.3
 L266.3,484.8 L264.4,484.3 L262.5,483.8 L260.7,483.2 L258.8,482.7 L257.0,482.2 L255.2,481.6 L253.4,481.1
 L251.6,480.6 L249.9,480.1 L248.1,479.5 L246.4,479.0 L244.7,478.5 L243.0,478.0 L241.3,477.4 L239.6,476.9
 L237.9,476.4 L236.3,475.8 L234.7,475.3 L233.0,474.8 L231.4,474.3 L229.8,473.7 L228.2,473.2 L226.7,472.7
 L225.1,472.1 L223.6,471.6 L222.0,471.1 L220.5,470.6 L219.0,470.0 L217.5,469.5 L216.0,469.0 L214.6,468.5
 L213.1,467.9 L211.7,467.4 L210.2,466.9 L208.8,466.3 L207.4,465.8 L206.0,465.3 L204.6,464.8 L203.2,464.2
 L201.9,463.7 L200.5,463.2 L199.2,462.6 L197.9,462.1 L196.5,461.6 L195.2,461.1 L193.9,460.5 L192.7,460.0
 L191.4,459.5 L190.1,458.9 L188.9,458.4 L187.6,457.9 L186.4,457.4 L185.2,456.8 L184.0,456.3 L182.8,455.8
 L181.6,455.3 L180.4,454.7 L179.2,454.2 L178.0,453.7 L176.9,453.1 L175.8,452.6 L174.6,452.1 L173.5,451.6
 L172.4,451.0 L171.3,450.5 L170.2,450.0 L169.1,449.4 L168.0,448.9 L167.0,448.4 L165.9,447.9 L164.9,447.3
 L163.8,446.8 L162.8,446.3 L161.8,445.8 L160.8,445.2 L159.8,444.7 L158.8,444.2 L157.8,443.6 L156.8,443.1
 L155.8,442.6 L154.9,442.1 L153.9,441.5 L153.0,441.0 L152.0,440.5 L151.1,439.9 L150.2,439.4 L149.3,438.9
 L148.4,438.4 L147.5,437.8 L146.6,437.3 L145.7,436.8 L144.8,436.2 L143.9,435.7 L143.1,435.2 L142.2,434.7
 L141

In [ ]:
double velW_top = -0.884473;
double velWstar_top = velW_top * Math.Sqrt(viscosity * omega); 
velWstar_top

-0.00884473

In [ ]:
Formula VelocityZ_top = new Formula(
    "VelZ",
    false,
    "double VelZ(double[] X) { return -0.00884172; } "
);

In [ ]:
List<XNSE_Control> Controls = new List<XNSE_Control>();
Controls.Clear();

In [ ]:
var C = new XNSE_Control();

int k = 3;
C.SetDGdegree(k);


// physical parameters
C.PhysicalParameters.rho_A = density;
C.PhysicalParameters.mu_A = density * viscosity;

C.PhysicalParameters.IncludeConvection = true;

    
// set grid
double l_radial = zstar / 10.0; 
double l_azimuthal = zstar / 5.0;
double h_axial = zstar; 

int res_global = 4;
int res_radial = 1 * res_global; 
int res_azimuthal = 2 * res_global;
int res_axial = 10 * res_global;

Grid2D grd = RotatingDisk2DSlice_NormalToRadial(l_azimuthal, h_axial, res_azimuthal, res_axial);
C.SetGrid(grd);

C.UseRotInertForceTerms = false;

// boundary condition
C.AddBoundaryValue("wall_rotatingDisk", "VelocityX#A", MovingWall);
C.AddBoundaryValue("pressure_dirichlet_top", "Pressure#A", vonKarman_P);
C.AddBoundaryValue("velocity_inlet_upstream", "VelocityX#A", vonKarman_velV);
// C.AddBoundaryValue("velocity_inlet_downstream", "VelocityX#A", vonKarman_velV);
C.AddBoundaryValue("pressure_dirichlet_downstream", "Pressure#A", vonKarman_P);


C.TimesteppingMode = AppControl._TimesteppingMode.Steady;
// C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
// C.NonLinearSolver.ConvergenceCriterion = 1e-9;
C.NonLinearSolver.MaxSolverIterations = 50;

// C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
// C.TimeSteppingScheme = TimeSteppingScheme.BDF3;
// C.dtFixed = Math.PI * 1.0e-3;
// C.NoOfTimesteps = 2000;
    
// {
//     C.AdaptiveMeshRefinement = true;
//     int AMRlevel = 1;
//     C.activeAMRlevelIndicators.Add(new AMRLevelIndicatorLibrary.AMRonBoundary(new byte[] { 1 }) { maxRefinementLevel = AMRlevel });
//     C.AMR_startUpSweeps = AMRlevel;
// }
    
C.SessionName = "RotatingDiskBoundaryLayer2D-TestRunSliceNormalToRadial6";
    
Controls.Add(C);

## Launch job

In [ ]:
Controls.Select(C => C.SessionName)

index,value
0,RotatingDiskBoundaryLayer2D-TestRunSliceNormalToRadial6


In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    oneJob.NumberOfMPIProcs = 1;
    oneJob.Activate(myBatch); 
}

## Postprocessing

In [ ]:
BoSSSshell.WorkflowMgm.Sessions

In [ ]:
OpenOrCreateDatabase(@"\\hpccluster\hpccluster-scratch\smuda\RotatingDiskBoundaryLayer2D");

Opening existing database '\\hpccluster\hpccluster-scratch\smuda\RotatingDiskBoundaryLayer2D'.


In [ ]:
databases.Pick(1).Sessions

#0: RotatingDiskBoundaryLayer2D	RotatingDiskBoundaryLayer2D-TestRunSliceNormalToRadial6	08/10/2023 16:46:23	eb67bd1a...
#1: RotatingDiskBoundaryLayer2D	RotatingDiskBoundaryLayer2D-TestRunSliceNormalToRadial5	08/10/2023 16:29:08	24ab91aa...
#2: RotatingDiskBoundaryLayer2D	RotatingDiskBoundaryLayer2D-TestRunSliceNormalToRadial4	08/10/2023 16:27:42	8fd8909e...
#3: RotatingDiskBoundaryLayer2D	RotatingDiskBoundaryLayer2D-TestRunSliceNormalToRadial3	08/10/2023 16:23:24	69a4d085...
#4: RotatingDiskBoundaryLayer2D	RotatingDiskBoundaryLayer2D-TestRunSliceNormalToRadial2	08/10/2023 16:18:54	56f24f5c...
#5: RotatingDiskBoundaryLayer2D	RotatingDiskBoundaryLayer2D-TestRunSliceNormalToRadial	08/10/2023 16:02:27	ec7cda23...


In [ ]:
// var sess = BoSSSshell.WorkflowMgm.Sessions.Pick(0);
var sess = databases.Pick(1).Sessions.Pick(0);
sess

RotatingDiskBoundaryLayer2D	RotatingDiskBoundaryLayer2D-TestRunSliceNormalToRadial6	08/10/2023 16:46:23	eb67bd1a...

In [ ]:
sess.Export().WithSupersampling(3).Do()

C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\RotatingDiskBoundaryLayer2D__RotatingDiskBoundaryLayer2D-TestRunSliceNormalToRadial6__eb67bd1a-e04c-49ed-86d5-91732a1eb8e1

### velocity profiles

In [ ]:
int tIdx = 1;
sess.Timesteps.Pick(tIdx)

 { Time-step: 1; Physical time: 1.7976931348623158E+304s; Fields: Phi, PhiDG, VelocityX, VelocityY, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, MPIrank, CutCells; Name:  }

In [ ]:
DGField[] solutionFields = new DGField[3];
solutionFields[0] = sess.Timesteps.Pick(tIdx).GetField("VelocityX");
solutionFields[1] = sess.Timesteps.Pick(tIdx).GetField("VelocityY");
solutionFields[2] = sess.Timesteps.Pick(tIdx).GetField("Pressure");

In [ ]:
double[][] solutionValues = new double[3][];
double[] evalPoint = new double[] {1.0};

// velocity fields
for (int j = 0; j < 3; j++) {
    solutionValues[j] = new double[numVal+1];
    for (int i = 0; i <= numVal; i++) {
        solutionValues[j][i] = solutionFields[j].ProbeAt(new double[] {evalPoint[0], zValues[i]});
    }
}
// pressure field (correct to same pressure level at wall p=0)
solutionValues[2] = new double[numVal+1];
double pressure0 = solutionFields[2].ProbeAt(new double[] {evalPoint[0], 0.0});
for (int i = 0; i <= numVal; i++) {
    solutionValues[2][i] = solutionFields[2].ProbeAt(new double[] {evalPoint[0], zValues[i]}) - pressure0;
}

In [ ]:
var fmt1 = new PlotFormat();    // blue - BoSSS
fmt1.LineColor = LineColors.Blue;
var fmt2 = new PlotFormat();    // red - Analytic

var gp = new Gnuplot();
gp.SetMultiplot(2,2);

int idx = 0;
int J = 2;
for (int i = 0; i < 2; i++) {
for (int j = 0; j < J; j++) {
    Plot2Ddata plt = new Plot2Ddata(); 
    plt.AddDataGroup("BoSSS", solutionValues[idx], zValues, fmt1);
    plt.AddDataGroup("Analytic", varValues[idx], zValues, fmt2);
    gp.SetSubPlot(i,j, varNames[idx]);
    plt.ToGnuplot(gp);
    idx++;
}
J--;
}
//gp.PlotNow()
gp.PlotSVG(xRes:1500,yRes:1000)

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 2 
 
 
 
 
 4 
 
 
 
 
 6 
 
 
 
 
 8 
 
 
 
 
 10 
 
 
 
 
 0 
 
 
 
 
 0.1 
 
 
 
 
 0.2 
 
 
 
 
 0.3 
 
 
 
 
 0.4 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 0.7 
 
 
 
 
 0.8 
 
 
 
 
 0.9 
 
 
 
 
 1 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 BoSSS 
 
 
 BoSSS 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M531.0,62.1 L584.4,62.1 M692.6,459.1 L688.6,458.7 L684.7,458.3 L680.7,457.8 L676.8,457.4 L672.8,457.0
 L668.9,456.6 L665.0,456.2 L661.1,455.8 L657.2,455.3 L653.3,454.9 L649.4,454.5 L645.5,454.1 L641.7,453.7
 L637.8,453.2 L634.0,452.8 L630.1,452.4 L626.3,452.0 L622.5,451.6 L618.7,451.2 L614.9,450.7 L611.1,450.3
 L607.4,449.9 L603.6,449.5 L599.9,449.1 L596.1,448.6 L592.4,448.2 L588.7,447.8 L585.0,447.4 L581.4,447.0
 L577.7,446.6 L574.0,446.1 L570.4,445.7 L566.8,445.3 L563.1,444.9 L559.5,444.5 L556.0,444.1 L552.4,443.6
 L548.8,443.2 L545.3,442.8 L541.7,442.4 L538.2,442.0 L534.7,441.5 L531.2,441.1 L527.8,440.7 L524.3,440.3
 L520.9,439.9 L517.5,439.5 L514.0,439.0 L510.7,438.6 L507.3,438.2 L503.9,437.8 L500.6,437.4 L497.2,436.9
 L493.9,436.5 L490.6,436.1 L487.4,435.7 L484.1,435.3 L480.9,434.9 L477.6,434.4 L474.4,434.0 L471.2,433.6
 L468.0,433.2 L464.9,432.8 L461.7,432.3 L458.6,431.9 L455.5,431.5 L452.4,431.1 L449.3,430.7 L446.3,430.3
 L443.2,429.8 L440.2,429.4 L437.2,429.0 L434.2,428.6 L431.3,428.2 L428.3,427.7 L425.4,427.3 L422.5,426.9
 L419.6,426.5 L416.7,426.1 L413.8,425.7 L411.0,425.2 L408.1,424.8 L405.3,424.4 L402.5,424.0 L399.8,423.6
 L397.0,423.2 L394.3,422.7 L391.6,422.3 L388.9,421.9 L386.2,421.5 L383.5,421.1 L380.9,420.6 L378.2,420.2
 L375.6,419.8 L373.0,419.4 L370.4,419.0 L367.9,418.6 L365.3,418.1 L362.8,417.7 L360.3,417.3 L357.8,416.9
 L355.3,416.5 L352.8,416.0 L350.4,415.6 L348.0,415.2 L345.6,414.8 L343.2,414.4 L340.8,414.0 L338.4,413.5
 L336.1,413.1 L333.8,412.7 L331.5,412.3 L329.2,411.9 L326.9,411.4 L324.6,411.0 L322.4,410.6 L320.2,410.2
 L318.0,409.8 L315.8,409.4 L313.6,408.9 L311.4,408.5 L309.3,408.1 L307.2,407.7 L305.0,407.3 L302.9,406.8
 L300.9,406.4 L298.8,406.0 L296.7,405.6 L294.7,405.2 L292.7,404.8 L290.7,404.3 L288.7,403.9 L286.7,403.5
 L284.8,403.1 L282.8,402.7 L280.9,402.3 L279.0,401.8 L277.1,401.4 L275.2,401.0 L273.3,400.6 L271.4,400.2
 L269.6,399.7 L267.8,399.3 L266.0,398.9 L264.2,398.5 L262.4,398.1 L260.6,397.7 L258.8,397.2 L257.1,396.8
 L255.4,396.4 L253.6,396.0 L251.9,395.6 L250.3,395.1 L248.6,394.7 L246.9,394.3 L245.3,393.9 L243.6,393.5
 L242.0,393.1 L240.4,392.6 L238.8,392.2 L237.2,391.8 L235.6,391.4 L234.1,391.0 L232.5,390.5 L231.0,390.1
 L229.5,389.7 L228.0,389.3 L226.5,388.9 L225.0,388.5 L223.5,388.0 L222.0,387.6 L220.6,387.2 L219.1,386.8
 L217.7,386.4 L216.3,385.9 L214.9,385.5 L213.5,385.1 L212.1,384.7 L210.7,384.3 L209.4,383.9 L208.0,383.4
 L206.7,383.0 L205.4,382.6 L204.1,382.2 L202.8,381.8 L201.5,381.4 L200.2,380.9 L198.9,380.5 L197.7,380.1
 L196.4,379.7 L195.2,379.3 L193.9,378.8 L192.7,378.4 L191.5,378.0 L190.3,377.6 L189.1,377.2 L187.9,376.8
 L186.8,376.3 L185.6,375.9 L184.4,375.5 L183.3,375.1 L182.2,374.7 L181.1,374.2 L179.9,373.8 L178.8,373.4
 L177.7,373.0 L176.7,372.6 L175.6,372.2 L174.5,371.7 L173.5,371.3 L172.4,370.9 L171.4,370.5 L170.3,370.1
 L169.3,369.6 L168.3,369.2 L167.3,368.8 L166.3,368.4 L165.3,368.0 L164.3,367.6 L163.4,367.1 L162.4,366.7
 L161.4,366.3 L160.5,365.9 L159.5,365.5 L158.6,365.0 L157.7,364.6 L156.8,364.2 L155.9,363.8 L155.0,363.4
 L154.1,363.0 L153.2,362.5 L152.3,362.1 L151.4,361.7 L150.6,361.3 L149.7,360.9 L148.9,360.5 L148.0,360.0
 L147.2,359.6 L146.4,359.2 L145.6,358.8 L144.7,358.4 L143.9,357.9 L143.1,357.5 L142.3,357.1 L141.6,356.7
 L140.8,356.